# Laboratorio 5: Carga y preprocesamiento de datos

**Objetivo:** Clasificar si un tweet se refiere a un desastre real (`target = 1`) o no (`target = 0`).

**Este notebook cubre:**
- Puntos 1 y 2: Descarga y carga del dataset `train.csv` desde Kaggle.
- Punto 3: Limpieza y preprocesamiento completo del texto.

---

In [1]:
import re
import string
import html

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid")

## 1. Carga de datos (Puntos 1 y 2)

Cargar el archivo `train.csv` descargado de la competencia de Kaggle ([Natural Language Processing with Disaster Tweets](https://www.kaggle.com/c/nlp-getting-started)).
El archivo debe colocarse en la carpeta `./data/` antes de ejecutar este notebook.

In [2]:
DATA_PATH = "./data/train.csv"
df = pd.read_csv(DATA_PATH)
df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are being notified by officers. No other evacuation or shelter in place or...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation orders in California",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as smoke from #wildfires pours into a school,1


## 2. Descripción general del dataset

Revisar las dimensiones, tipos de datos y valores nulos por columna.

In [3]:
print(f"Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}")
df.info()

Filas: 7,613  |  Columnas: 5
<class 'pandas.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        7613 non-null   int64
 1   keyword   7552 non-null   str  
 2   location  5080 non-null   str  
 3   text      7613 non-null   str  
 4   target    7613 non-null   int64
dtypes: int64(2), str(3)
memory usage: 1.2 MB


In [4]:
nulos = df.isnull().sum().to_frame("nulos")
nulos["% del total"] = (nulos["nulos"] / len(df) * 100).round(2)
nulos

,nulos,% del total
id,0,0.00
keyword,61,0.80
location,2533,33.27
text,0,0.00
target,0,0.00


**Análisis:**
- `id`, `text` y `target` no tienen valores nulos, esto indica que son las columnas núcleo del problema.
- `keyword` tiene relativamente pocos nulos con menos del 1%, lo que significa que es utilizable como variable auxiliar.
- `location` tiene una proporción alta de nulos alrededor del 33%, por lo que no será tomada en cuenta para este análisis.

## 3. Preprocesamiento y limpieza de texto (Punto 3)

Antes de analizar frecuencias de palabras, generar n-gramas o entrenar un modelo, el texto
crudo de Twitter necesita limpieza, ya que contiene URLs, menciones (`@usuario`), hashtags (`#tema`),
entidades HTML `&amp;`, errores de codificación y puntuación o números que
generalmente no aportan valor semántico.

In [5]:
pd.set_option("display.max_colwidth", 200)
df[["text"]].sample(6, random_state=42)

,text
2644,So you have a new weapon that can cause un-imaginable destruction.
2227,The f$&amp;@ing things I do for #GISHWHES Just got soaked in a deluge going for pads and tampons. Thx @mishacollins @/@
5448,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe CoL police can catch a pickpocket in Liverpool Stree... http://t.co/vXIn1gOq4Q
132,Aftershock back to school kick off was great. I want to thank everyone for making it possible. What a great night.
6845,in response to trauma Children of Addicts develop a defensive self - one that decreases vulnerability. (3
5559,@Calum5SOS you look like you got caught in a rainstorm this is amazing and disgusting at the same time


### 3.1 Problema: caracteres mal codificados

Al revisar tweets al azar se detectaron secuencias como `\x89ÛÒ`, `\x89Ûª` o `åÊ`. Esto ocurre
porque el dataset original tenía texto en UTF-8 que fue re-codificado incorrectamente como Latin-1/CP1252.

In [6]:
patron_mojibake = df["text"].str.contains(r"\\x89|Û|åÊ|åÈ", regex=True, na=False)
print(f"Tweets con artefactos de codificación: {patron_mojibake.sum()} "
      f"({patron_mojibake.mean()*100:.1f}% del total)")
df.loc[patron_mojibake, "text"].head(3).tolist()

Tweets con artefactos de codificación: 658 (8.6% del total)


['Barbados #Bridgetown JAMAICA \x89ÛÒ Two cars set ablaze: SANTA CRUZ \x89ÛÓ Head of the St Elizabeth Police Superintende...  http://t.co/wDUEaj8Q4J',
 'SANTA CRUZ \x89ÛÓ Head of the St Elizabeth Police Superintendent Lanford Salmon has r ... - http://t.co/vplR5Hka2u http://t.co/SxHW2TNNLf',
 'Police: Arsonist Deliberately Set Black Church In North CarolinaåÊAblaze http://t.co/pcXarbH9An']

### 3.2 Verificación de emojis

In [7]:
# Verificación explícita de emojis Unicode reales en el texto crudo
patron_emoji = re.compile(
    "["
    "\U0001F600-\U0001F64F"  # emoticones
    "\U0001F300-\U0001F5FF"  # símbolos y pictogramas
    "\U0001F680-\U0001F6FF"  # transporte y mapas
    "\U0001F900-\U0001F9FF"  # símbolos suplementarios
    "\U00002600-\U000026FF"  # símbolos varios
    "\U00002700-\U000027BF"  # dingbats
    "]+", flags=re.UNICODE
)

tweets_con_emoji = df["text"].apply(lambda t: len(patron_emoji.findall(t)) > 0).sum()
print(f"Tweets con emojis Unicode reales detectados: {tweets_con_emoji}")

Tweets con emojis Unicode reales detectados: 0


### 3.3 Stopwords

In [8]:
stop_words_en = set(stopwords.words("english"))
print(f"Número de stopwords en inglés (nltk): {len(stop_words_en)}")
list(stop_words_en)[:15]

Número de stopwords en inglés (nltk): 198


['whom',
 'any',
 'too',
 'his',
 "hasn't",
 'they',
 'at',
 'more',
 'were',
 'needn',
 'by',
 'after',
 'these',
 'won',
 'yourself']

### 3.4 Diseño del pipeline de limpieza

Se aplican los pasos en un orden pensado para no generar efectos secundarios:

1. **Corregir artefactos de codificación** (`\x89Û_`, `åÊ`, etc.)
2. **Quitar URLs** (`http://...`, `https://...`, `www...`)
3. **Decodificar entidades HTML** (`&amp;` → `&`, etc.)
4. **Quitar menciones** (`@usuario`)
5. **Procesar hashtags** (`#earthquake` → `earthquake`)
6. **Convertir a minúsculas**
7. **Quitar signos de puntuación y caracteres especiales**
8. **Quitar números, con una excepción explícita: `911`** — se conserva por su fuerte carga semántica de emergencia.
9. **Tokenización**
10. **Quitar *stopwords***

In [9]:
def corregir_mojibake(texto):
    texto = re.sub(r"\x89Û_", "'", texto)
    texto = re.sub(r"\x89Û[ÒÓªÏ]", " ", texto)
    texto = re.sub(r"åÊ|åÈ", " ", texto)
    texto = re.sub(r"[\x80-\x9f]", " ", texto)
    return texto


def quitar_urls(texto):
    return re.sub(r"https?://\S+|www\.\S+", " ", texto)


def quitar_menciones(texto):
    return re.sub(r"@\w+", " ", texto)


def procesar_hashtags(texto):
    # Quita el símbolo '#' pero conserva la palabra, ya que suele ser informativa
    return re.sub(r"#(\w+)", r"\1", texto)


def quitar_numeros_excepto_911(texto):
    tokens = texto.split()
    tokens = [t for t in tokens if not (t.isdigit() and t != "911")]
    return " ".join(tokens)


def limpiar_tweet(texto):
    texto = corregir_mojibake(texto)
    texto = quitar_urls(texto)
    texto = html.unescape(texto)
    texto = quitar_menciones(texto)
    texto = procesar_hashtags(texto)
    texto = texto.lower()
    texto = re.sub(f"[{re.escape(string.punctuation)}]", " ", texto)
    texto = quitar_numeros_excepto_911(texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    tokens = [w for w in texto.split() if w not in stop_words_en and len(w) > 1]
    return " ".join(tokens)


# Ejemplo paso a paso sobre un tweet real del dataset
ejemplo = df.loc[df["text"].str.contains("earthquake", case=False), "text"].iloc[0]
print("Original      :", ejemplo)
print("Sin URLs      :", quitar_urls(ejemplo))
print("Sin menciones :", quitar_menciones(quitar_urls(ejemplo)))
print("Limpio final  :", limpiar_tweet(ejemplo))

Original      : Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all
Sin URLs      : Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all
Sin menciones : Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all
Limpio final  : deeds reason earthquake may allah forgive us


In [10]:
df["clean_text"] = df["text"].apply(limpiar_tweet)
df[["text", "clean_text"]].sample(8, random_state=7)

,text,clean_text
6106,If there's a chance will get a gander of the sinking ship that is #TNA too. Can't help but appease my morbid curiosity. #DestinationIMPACT,chance get gander sinking ship tna help appease morbid curiosity destinationimpact
5025,2 great new recipes; mudslide cake and so sorry stew! #GBBO,great new recipes mudslide cake sorry stew gbbo
7022,#breaking #news Global precipitation measurement satellite captures 3-D image of Typhoon Soudelor - @NASAHurricane http://t.co/20DNcthr4D,breaking news global precipitation measurement satellite captures image typhoon soudelor
2331,EPA begins demolition of homes in toxic area #Buffalo - http://t.co/noRkXBRS6G,epa begins demolition homes toxic area buffalo
3354,KATUNews: #SR14 remains closed as brush fire burns 1700 acres: http://t.co/QposKp3MWj #LiveOnK2 http://t.co/mTQjsvupwy,katunews sr14 remains closed brush fire burns acres liveonk2
6117,Spent too many hours sinking into the wonderfully created worlds of Mafia and Mafia II in my life. Excited for another installment.,spent many hours sinking wonderfully created worlds mafia mafia ii life excited another installment
2258,Businesses are deluged with invoices. Make yours stand oup with colour or shame and it's likely to rise to the top of the pay' pile.,businesses deluged invoices make stand oup colour shame likely rise top pay pile
2801,@Hollyorange8 my day has been a disaster of emotions,day disaster emotions


### 3.5 Validación del preprocesamiento

Antes de continuar, se revisa que la limpieza no haya generado efectos indeseados.

In [11]:
df["n_caracteres"] = df["text"].str.len()
df["n_palabras"] = df["text"].str.split().apply(len)
df["clean_n_palabras"] = df["clean_text"].str.split().apply(len)

vacios = df[df["clean_n_palabras"] == 0]
print(f"Tweets que quedaron vacíos tras la limpieza: {len(vacios)}")
vacios[["text", "target"]]

Tweets que quedaron vacíos tras la limpieza: 2


,text,target
4497,@Hurricane_Dame ???????? I don't have them they out here,1
6766,@Ayshun_Tornado then don't,0


Solo un puñado de tweets queda vacío tras la limpieza, ya que eran mensajes que consistían básicamente en una mención
a otro usuario más una o dos *stopwords*. Se decide **conservarlos** tal cual en vez de eliminarlos, ya que descartar filas cambiaría el balance de clases y el vectorizador simplemente los representará como un vector de ceros.

In [12]:
comparacion_longitud = pd.DataFrame({
    "promedio_palabras_original": [df["n_palabras"].mean()],
    "promedio_palabras_limpio": [df["clean_n_palabras"].mean()],
    "reduccion_%": [(1 - df["clean_n_palabras"].mean() / df["n_palabras"].mean()) * 100]
}).round(2)
comparacion_longitud

,promedio_palabras_original,promedio_palabras_limpio,reduccion_%
0,14.9,8.62,42.17


La limpieza reduce la longitud promedio de los tweets en más de un tercio, principalmente por la eliminación de *stopwords*, menciones y URLs.

## 4. Guardado del dataset preprocesado

Se guarda el dataset con la columna `clean_text` en un nuevo CSV (`train_clean.csv`), que será el punto de partida de los notebooks siguientes, evitando repetir el preprocesamiento.

In [13]:
columnas_finales = ["id", "keyword", "location", "text", "clean_text",
                    "n_caracteres", "n_palabras", "clean_n_palabras", "target"]
df[columnas_finales].to_csv("./data/train_clean.csv", index=False)
print("Archivo './data/train_clean.csv' guardado con", df.shape[0], "filas.")

Archivo './data/train_clean.csv' guardado con 7613 filas.


## 5. Resumen

- El dataset tiene **7,613 tweets** con un desbalance moderado de clases (**57% no-desastre / 43% desastre real**).
- El texto crudo requería limpieza en varios frentes: URLs, menciones, hashtags, entidades HTML, artefactos de codificación (*mojibake*, ~8.6% de los tweets) y *stopwords*.
- Se decidió **conservar el token `911`** por su fuerte carga semántica relacionada con emergencias.
- No se detectaron emoticones Unicode reales en el dataset (verificado explícitamente).
- El dataset limpio (`train_clean.csv`) queda listo para el análisis de frecuencias y n-gramas en el **Notebook 02**.